# 🎬 AI 影片生成工作流 (Google Colab Notebook)

本筆記本提供完整的 AI 影音生成工作流，支援：
- 🎨 **旗艦生圖 (Soul v2)**：生成高品質、擬真光影的人物或場景首幀（Keyframe）。
- 🎥 **電影級運鏡影片生成 (Cinematic Studio)**：支援 **文字轉影片 (Text-to-Video)** 與 **圖片轉影片 (Image-to-Video)**。
- 💾 **預覽與儲存**：直接於 Colab 內預覽生成影片，並支援儲存至 Google Drive 或本機下載。

## 步驟 1：安裝必要套件
點擊執行以下儲存格以安裝 `higgsfield-client` 與相關工具庫。

In [ ]:
# 安裝必要套件
!pip install -q higgsfield-client python-dotenv requests ipywidgets

## 步驟 2：設定 API Key 與工作目錄
- 建議在 Colab 左側鑰匙圖示 🔑 (Secrets) 中新增 `HF_KEY`。
- 或者直接在下方輸入你的 API Key。

In [ ]:
import os
from pathlib import Path

# 嘗試從 Colab Secrets 讀取，若無則手動輸入
try:
    from google.colab import userdata
    hf_key = userdata.get('HF_KEY')
except Exception:
    hf_key = None

if not hf_key:
    import getpass
    hf_key = getpass.getpass('請輸入 Higgsfield HF_KEY (格式 key_id:key_secret): ')

os.environ['HF_KEY'] = hf_key.strip()
print('✅ API 金鑰設定完成！')

# 建立輸出資料夾
OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'📁 輸出路徑已準備就緒: {OUTPUT_DIR.resolve()}')

## 步驟 3：(可選) 掛載 Google Drive
若希望影片自動備份到個人雲端硬碟，請執行此儲存格。

In [ ]:
# 掛載 Google Drive (如不需要可跳過)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE_PATH = Path('/content/drive/MyDrive/AI_Videos')
DRIVE_SAVE_PATH.mkdir(parents=True, exist_ok=True)
print(f'📁 雲端硬碟備份路徑: {DRIVE_SAVE_PATH}')

## 步驟 4：文字生圖 (Soul v2 旗艦生圖)
生成作為影片起始幀的超高畫質擬真畫面。

In [ ]:
import higgsfield_client
import requests
import time
from IPython.display import Image, display

prompt_image = "Cinematic portrait of a cyberpunk wanderer in a rainy neon alley, soft volumetric lights, photorealistic, 4k, 85mm lens"
aspect_ratio = "16:9" # 選項: 16:9, 9:16, 1:1, 4:3

print("⏳ 正在生成起始圖像...")
image_res = higgsfield_client.subscribe(
    "higgsfield-ai/soul/v2/standard",
    arguments={
        "prompt": prompt_image,
        "aspect_ratio": aspect_ratio,
        "resolution": "2K"
    }
)

image_url = image_res["images"][0]["url"]
print("✅ 圖像生成成功！URL:", image_url)

# 下載並顯示
img_filename = OUTPUT_DIR / f"frame_{int(time.time())}.png"
with open(img_filename, "wb") as f:
    f.write(requests.get(image_url).content)

display(Image(filename=str(img_filename)))

## 步驟 5：動態影片生成 (Cinematic Studio 運鏡)
可選擇 **以圖生影 (Image-to-Video)** 或 **純文字生影 (Text-to-Video)**。

In [ ]:
from IPython.display import HTML
import shutil

# 設定運鏡提示詞與參數
video_prompt = "Slow forward dolly shot, rain dripping from neon signs, gentle breath mist, atmospheric lighting"
duration = 5 # 秒數 (預設 5)

print("🎬 正在生成動態運鏡影片 (約需 1~3 分鐘)...")
video_args = {
    "prompt": video_prompt,
    "aspect_ratio": aspect_ratio,
    "duration": duration,
    "image_url": image_url # 若要純文字生影，可註解此行
}

video_res = higgsfield_client.subscribe(
    "higgsfield-ai/cinematic_studio_2_5",
    arguments=video_args
)

# 解析影片網址
if isinstance(video_res, dict):
    video_url = video_res.get("video_url") or video_res.get("url")
    if not video_url and "videos" in video_res and len(video_res["videos"]) > 0:
        video_url = video_res["videos"][0].get("url")
else:
    video_url = str(video_res)

print("🎉 影片生成完成！URL:", video_url)

# 下載影片至本地
video_filename = OUTPUT_DIR / f"video_{int(time.time())}.mp4"
with open(video_filename, "wb") as f:
    f.write(requests.get(video_url).content)
print(f"💾 已儲存至: {video_filename}")

# 若有掛載 Drive 則複製一份備份
if 'DRIVE_SAVE_PATH' in globals() and DRIVE_SAVE_PATH.exists():
    drive_file = DRIVE_SAVE_PATH / video_filename.name
    shutil.copyfile(video_filename, drive_file)
    print(f"☁️ 已同步備份至 Google Drive: {drive_file}")

# 於 Colab 內嵌播放影片
HTML(f'''
<video width="640" height="360" controls autoplay loop>
  <source src="{video_url}" type="video/mp4">
  您的瀏覽器不支援 HTML5 影片播放。
</video>
''')